# Baseline Colab Run

This notebook runs the baseline (no SSL) training and evaluation pipeline on Google Colab with GPU acceleration.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from datetime import datetime

REPO_URL = 'https://github.com/satvikkaul/SSL_Prostate_Cancer_Grading.git'
BRANCH = 'method/moco-v2'  # Use the same branch as MoCo
PROJECT_DIR = '/content/SSL_Prostate_Cancer_Grading'
DRIVE_ROOT = '/content/drive/MyDrive/Prostate_SSL'
DATASET_ZIP = f'{DRIVE_ROOT}/dataset.zip'
RUN_NAME = f'baseline_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
DRIVE_RUN_DIR = f'{DRIVE_ROOT}/runs/{RUN_NAME}'
os.environ['MPLCONFIGDIR'] = '/tmp/mplconfig'

print('PROJECT_DIR =', PROJECT_DIR)
print('DATASET_ZIP =', DATASET_ZIP)
print('DRIVE_RUN_DIR =', DRIVE_RUN_DIR)

In [ ]:
%cd /content
!rm -rf "$PROJECT_DIR"
!git clone -b "$BRANCH" "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"
!git branch --show-current
!git log -1 --oneline

## Install Dependencies

Colab already has TensorFlow installed, so we only install additional requirements.

In [ ]:
%cd "$PROJECT_DIR"
!pip install -q openpyxl scikit-learn pandas matplotlib pillow opencv-python

## Setup Dataset

Extract dataset.zip from Drive to local Colab storage and generate CSV splits.

In [ ]:
%cd "$PROJECT_DIR"
import shutil

if os.path.exists('./dataset'):
    shutil.rmtree('./dataset')

!unzip -q "$DATASET_ZIP" -d .
!python data/setup.py

## Link Output to Drive

Symlink ./output to the Drive-backed run folder so all artifacts are automatically saved to Drive.

In [ ]:
%cd "$PROJECT_DIR"
import shutil

if os.path.exists('./output'):
    shutil.rmtree('./output')

os.makedirs(DRIVE_RUN_DIR, exist_ok=True)
os.symlink(DRIVE_RUN_DIR, './output')
print(f'Linked ./output -> {DRIVE_RUN_DIR}')

## Apply Patches

Add CLI arguments to train_baseline.py for epochs and batch size control.

In [ ]:
%cd "$PROJECT_DIR"
import re

baseline_path = 'training/baseline/train_baseline.py'
with open(baseline_path, 'r') as f:
    content = f.read()

# Check if already patched
if 'argparse' not in content:
    # Add argparse import after sys import
    content = content.replace(
        'import sys',
        'import sys\nimport argparse'
    )
    
    # Find the CONFIGURATION section and add argument parser before it
    config_pattern = r'(# ={70,}\n# CONFIGURATION\n# ={70,})'
    parser_code = '''
def parse_args():
    parser = argparse.ArgumentParser(description="Baseline training (no SSL)")
    parser.add_argument('--epochs', type=int, default=50, help='Number of training epochs')
    parser.add_argument('--batch_size', type=int, default=8, help='Batch size')
    parser.add_argument('--lr', type=float, default=0.00001, help='Learning rate')
    return parser.parse_args()

args = parse_args()

'''
    content = re.sub(config_pattern, parser_code + r'\1', content)
    
    # Replace hardcoded values with args
    content = content.replace('BATCH_SIZE = 8', 'BATCH_SIZE = args.batch_size')
    content = content.replace('EPOCHS = 50', 'EPOCHS = args.epochs')
    content = content.replace('LEARNING_RATE = 0.00001', 'LEARNING_RATE = args.lr')
    
    with open(baseline_path, 'w') as f:
        f.write(content)
    
    print('✓ Patched train_baseline.py with CLI arguments')
else:
    print('✓ train_baseline.py already patched')

## Train Baseline

Train classifier from scratch with random initialization. Increase epochs for better results.

In [ ]:
%cd "$PROJECT_DIR"
EPOCHS = 100        # Increase from default 50
BATCH_SIZE = 32     # T4 can handle 32, A100 can try 64
LR = 0.00001        # Default learning rate

cmd = f'python training/baseline/train_baseline.py --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR}'
print(cmd)
!$cmd

## Evaluate Baseline

Generate evaluation metrics on the test set.

In [ ]:
%cd "$PROJECT_DIR"
!python evaluation/baseline/eval_baseline.py

## Sync Runtime State to Drive

Copy additional runtime artifacts to Drive.

In [ ]:
%cd "$PROJECT_DIR"

import shutil
from pathlib import Path

sync_root = Path(DRIVE_RUN_DIR) / "runtime_sync"
dataset_sync = sync_root / "dataset"
notebook_sync = sync_root / "notebook"

dataset_sync.mkdir(parents=True, exist_ok=True)
notebook_sync.mkdir(parents=True, exist_ok=True)

# Copy notebook if it exists in project dir (optional)
if Path("run_baseline_colab.ipynb").exists():
    shutil.copy2("run_baseline_colab.ipynb", notebook_sync / "run_baseline_colab.ipynb")

for name in ["Train.csv", "Test.csv", "TrainSplit.csv", "Val.csv"]:
    src = Path("dataset") / name
    if src.exists():
        shutil.copy2(src, dataset_sync / name)

os.system(f'git rev-parse HEAD > "{sync_root / "commit.txt"}"')
os.system(f'find ./output -maxdepth 4 -type f | sort > "{sync_root / "output_manifest.txt"}"')

print(f"Synced runtime artifacts to: {sync_root}")

In [ ]:
%cd "$PROJECT_DIR"
!echo "Run directory: $DRIVE_RUN_DIR"
!find ./output -maxdepth 3 -type f | sort | tail -n 20